In [1]:
import pandas as pd
import joblib

DATA_DIR = '/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/'

train = pd.read_csv(DATA_DIR + 'train_features.csv')
val = pd.read_csv(DATA_DIR + 'val_features.csv')
test = pd.read_csv(DATA_DIR + 'test_features.csv')

with open(DATA_DIR + 'feature_list.txt') as f:
    feature_list = f.read().splitlines()

X_train, y_train = train[feature_list], train['is_late']
X_val, y_val = val[feature_list], val['is_late']
X_test, y_test = test[feature_list], test['is_late']

print(X_train.shape, X_val.shape, X_test.shape)

(57323, 39) (20550, 39) (18603, 39)


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

val_pred_baseline = baseline.predict(X_val)
print("Baseline (always predict on-time) - Validation performance:")
print(classification_report(y_val, val_pred_baseline))

Baseline (always predict on-time) - Validation performance:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     18137
           1       0.00      0.00      0.00      2413

    accuracy                           0.88     20550
   macro avg       0.44      0.50      0.47     20550
weighted avg       0.78      0.88      0.83     20550



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

val_pred = logreg.predict(X_val)
val_proba = logreg.predict_proba(X_val)[:, 1]

print("Logistic Regression - Validation performance:")
print(classification_report(y_val, val_pred))
print("ROC-AUC:", roc_auc_score(y_val, val_proba))

Logistic Regression - Validation performance:
              precision    recall  f1-score   support

           0       0.93      0.65      0.76     18137
           1       0.19      0.61      0.29      2413

    accuracy                           0.65     20550
   macro avg       0.56      0.63      0.53     20550
weighted avg       0.84      0.65      0.71     20550

ROC-AUC: 0.6931595894863017


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

val_pred_rf = rf.predict(X_val)
val_proba_rf = rf.predict_proba(X_val)[:, 1]

print("Random Forest - Validation performance:")
print(classification_report(y_val, val_pred_rf))
print("ROC-AUC:", roc_auc_score(y_val, val_proba_rf))

Random Forest - Validation performance:
              precision    recall  f1-score   support

           0       0.91      0.86      0.89     18137
           1       0.27      0.39      0.32      2413

    accuracy                           0.81     20550
   macro avg       0.59      0.63      0.60     20550
weighted avg       0.84      0.81      0.82     20550

ROC-AUC: 0.6826586778015765


In [5]:
importances = pd.Series(rf.feature_importances_, index=feature_list).sort_values(ascending=False)
print(importances.head(15))

purchase_month        0.206353
total_freight         0.118379
estimated_days        0.115988
customer_state_RJ     0.085574
customer_state_SP     0.075895
total_payment         0.071514
total_price           0.066015
customer_state_MG     0.048526
purchase_hour         0.043804
purchase_weekday      0.031607
max_installments      0.027860
customer_state_PR     0.016842
customer_state_AL     0.012755
n_items               0.011846
n_distinct_sellers    0.008609
dtype: float64


In [6]:
test_pred = logreg.predict(X_test)
test_proba = logreg.predict_proba(X_test)[:, 1]

print("FINAL TEST SET EVALUATION (touched once):")
print(classification_report(y_test, test_pred))
print("ROC-AUC:", roc_auc_score(y_test, test_proba))


FINAL TEST SET EVALUATION (touched once):
              precision    recall  f1-score   support

           0       0.98      0.53      0.69     17584
           1       0.09      0.84      0.17      1019

    accuracy                           0.55     18603
   macro avg       0.54      0.68      0.43     18603
weighted avg       0.93      0.55      0.66     18603

ROC-AUC: 0.7088474411566943


In [7]:
results_summary = """
notebook 6 - model results summary

baseline (always predict on-time): 88% accuracy but 0.00 recall/precision 
on the late class, basically useless, never even predicts late once

logistic regression (class_weight=balanced):
  validation: roc-auc 0.693, recall 0.61, precision 0.19
  test:       roc-auc 0.709, recall 0.84, precision 0.09

random forest (class_weight=balanced, n_estimators=200, max_depth=10):
  validation: roc-auc 0.683, recall 0.39, precision 0.27

model i went with: logistic regression, picked recall over precision 
since missing an actual late order is worse than a false alarm for 
this use case

precision dropped a lot on test vs val (0.19 -> 0.09) while recall went 
up (0.61 -> 0.84). this isnt the model getting worse, its because the 
late rate itself changed between periods (val was 11.7% late, test was 
only 5.5% late, found this back in notebook 3). roc-auc actually held up 
fine (0.693 -> 0.709 even a bit better) which means the model still 
ranks late vs on-time orders correctly on new data, the precision drop 
is just from the class balance shifting

top features (from random forest importance): purchase_month, 
total_freight, estimated_days, customer_state (RJ/SP), total_payment, 
total_price - matches what i found in the eda in notebook 4

limitations / next steps:
- precision is pretty low so this isnt ready to be a hard yes/no alert, 
  more useful as a risk score to prioritize orders than a strict flag
- could try tuning the threshold, or xgboost, or add more features like 
  seller location/distance later on
"""

with open('/kaggle/working/results_summary_notebook6.txt', 'w') as f:
    f.write(results_summary)

print(results_summary)


notebook 6 - model results summary

baseline (always predict on-time): 88% accuracy but 0.00 recall/precision 
on the late class, basically useless, never even predicts late once

logistic regression (class_weight=balanced):
  validation: roc-auc 0.693, recall 0.61, precision 0.19
  test:       roc-auc 0.709, recall 0.84, precision 0.09

random forest (class_weight=balanced, n_estimators=200, max_depth=10):
  validation: roc-auc 0.683, recall 0.39, precision 0.27

model i went with: logistic regression, picked recall over precision 
since missing an actual late order is worse than a false alarm for 
this use case

precision dropped a lot on test vs val (0.19 -> 0.09) while recall went 
up (0.61 -> 0.84). this isnt the model getting worse, its because the 
late rate itself changed between periods (val was 11.7% late, test was 
only 5.5% late, found this back in notebook 3). roc-auc actually held up 
fine (0.693 -> 0.709 even a bit better) which means the model still 
ranks late vs on-t

In [8]:
import joblib, os, sklearn
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA_DIR = '/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/'
train = pd.read_csv(DATA_DIR + 'train_features.csv')
val = pd.read_csv(DATA_DIR + 'val_features.csv')
with open(DATA_DIR + 'feature_list.txt') as f:
    feature_list = f.read().splitlines()

# Final model: logistic regression, class_weight balanced (chosen in this notebook)
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(train[feature_list], train['is_late'])

print("val ROC-AUC:", roc_auc_score(val['is_late'], logreg.predict_proba(val[feature_list])[:, 1]))

# Save the model artifact
joblib.dump(logreg, '/kaggle/working/final_model_logreg.pkl')
print("sklearn:", sklearn.__version__)
print(os.listdir('/kaggle/working'))

val ROC-AUC: 0.6931595894863017
sklearn: 1.6.1
['results_summary_notebook6.txt', '__notebook__.ipynb', 'final_model_logreg.pkl']
